# Execute metagenome functional profile
Jacobo de la Cuesta-Zuluaga. August 2026.

The aim of this notebook is to obtain the functional profile of metagenome samples.
For this, we'll use `mifaser`. It assigns reads to a given Enzyme Commission (EC)
number using a curated database without the need of metagenome assembly.

The documentation of mifaser is available 
[here](https://bitbucket.org/bromberglab/mifaser/src/master/). If you use it,
remember to cite the mifaser [paper](https://doi.org/10.1093/nar/gkx1209).

## Before we start

The present notebook will continue using the sequence files we used in the `Sequence_QC`
notebook. Run notebook 01 first and wait until the QC job has finished. 
You should have `merged.R*.fastq.gz` files in `data/detaxizer/concatenated`.
Alternatively, you can use the output of detaxizer directly. These `filtered.fastq.gz`
files are in `detaxizer/filter/filtered`, and a sample sheet should be available in
`detaxizer/downstream_samplesheets`.

This notebook requires `conda` and the `Profiling` and `VScode` environments of this repo.
Instructions to install `conda` are [here](https://conda.io/projects/conda/en/latest/user-guide/install/index.html).
If you are on the M3 cluster you should have conda available and if you executed the
`Sequence_QC` notebook, you should have the environments already configured. 


You can install the required environments once with:

```bash
cd Path/To/Metemgee
conda env create -f envs/Profiling.yaml
conda env create -f envs/VScode.yaml
```
The notebooks are written in **R**, not Python. In VSCode, click the kernel selector on the top right
and pick the R kernel from the `VScode` environment.

## What you'll need to change

These are the only values you have to edit. Everything else can stay as it is.

| Variable | Where | What to put there |
|---|---|---|
| `base_dir` | Load libraries and set paths | The same one you used in notebook 01 |
| `seq_dir` | Load libraries and set paths | Only if your reads were not processed with `detaxizer` |
| `-d GS-24-all` | Perform functional profiling | Only if you want to use a different `mifaser` database |
| `cpu`, `memory` | Perform functional profiling | Only if your samples are much deeper than the example |

## Load libraries and set paths

First, we'll set up the libraries and the work directory where we'll save our files.

In [1]:
# Libraries
library(tidyverse)
library(conflicted)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.1.0     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the ]8;;http://conflicted.r-lib.org/conflicted package]8;; to force all conflicts to become errors


In [2]:
# Housekeeping: tells R which `filter` function to use when more than one
# package provides one. Nothing to change here.
conflicts_prefer(dplyr::filter)

[conflicted] Will prefer dplyr::filter over any other package.


The following chunk will define the directories where the data is stored and where the output will be
saved. The present example assumes everything will be contained in the same directory: `base_dir`.
This might be different in your particular case, for example, if your sequences are stored on a
centralized directory or you have multiple runs stored in different folders. You can change this
accordingly.

`base_dir` has to exist already; in this case it is the same we defined in the
`Sequence_QC` notebook.

If a folder already exists, `dir.create` prints a warning. That's harmless.

In [ ]:
# Directories
# Base directory
base_dir <- "/mnt/lustre/groups/maier/maide581/projects/Small_projects/Metemgee_remake"

# Data
data_dir <- file.path(base_dir, "data")
dir.create(data_dir)

# Output dir
mifaser_dir = file.path(data_dir, "mifaser")
dir.create(mifaser_dir)

# Sheets dir
sheets_dir <- file.path(data_dir, "sheets")
dir.create(sheets_dir)

# Detaxizer outputs
# Change if using sequences processed elsewhere
seq_dir <- file.path(data_dir, "detaxizer/concatenated")

# Software
conda_env <- "Profiling"
bin_dir = file.path(base_dir, "bin")
dir.create(bin_dir)

Warning message:
In dir.create(data_dir) :
  '/mnt/lustre/groups/maier/maide581/projects/Small_projects/Metemgee_remake/data' already exists
Warning message:
In dir.create(mifaser_dir) :
  '/mnt/lustre/groups/maier/maide581/projects/Small_projects/Metemgee_remake/data/mifaser' already exists
Warning message:
In dir.create(sheets_dir) :
  '/mnt/lustre/groups/maier/maide581/projects/Small_projects/Metemgee_remake/data/sheets' already exists
Warning message:
In dir.create(bin_dir) :
  '/mnt/lustre/groups/maier/maide581/projects/Small_projects/Metemgee_remake/bin' already exists


In [4]:
# Check that the base and sequence directories are defined and exist
stopifnot(dir.exists(base_dir), dir.exists(seq_dir))

## Download functional profiling software

Next, we need to download `mifaser`, which is the software we'll use to obtain
the functional profile from the metagenomes. We'll retrieve the complete 
repository, which comes with all the additional software and databases required.

In [5]:
# mifaser repo
mifaser_url <- "https://bitbucket.org/bromberglab/mifaser.git"
mifaser_repo <- file.path(bin_dir, "mifaser")

# Check if repo exists, if not, downloads it
if (!file.exists(file.path(mifaser_repo, "README.md"))) {
  # Download mifaser repo
  print("Cloning mifaser repository")
  str_glue(
    "git clone https://bitbucket.org/bromberglab/mifaser.git {mifaser_repo}",
    mifaser_repo = mifaser_repo
  ) |>
    system()
} else {
  print("Repo already exists")
}

[1] "Repo already exists"


## Samples file

In the present example we will use the clean concatenated files generated after
running `detaxizer`. 

Similar to the file we passed to `taxprofiler`, we'll need to create a `csv` file
with the name of the sample and the files corresponding to forward and reverse reads.
Importantly, this file needs to have a first column called `ArrayTaskID` with the
number of the sample (1 for first sample, 2 for second and so on).

The fields of this table are:

| Column | What goes in it |
|---|---|
| `ArrayTaskID` | Row number, starting at 1. This is how the cluster knows which sample each task should work on |
| `sample` | Sample name. It becomes the name of the output folder, so avoid spaces and special characters |
| `fastq_1`, `fastq_2` | Full path of the forward and reverse files |

If you have multiple `detaxizer` runs, you can load and combine the tables to
perform a single `mifaser` run.

In [6]:
# List clean sequences
clean_seq_list <- list.files(seq_dir, pattern = "fastq.gz", full.names = TRUE)

# Forward reads
forward_reads <- clean_seq_list |> 
  str_subset("R1")
# Reverse reads
reverse_reads <- clean_seq_list |> 
  str_subset("R2")

In [7]:
# Create a single data frame for mifaser
samples_table <- data.frame(
  fastq_1 = forward_reads, 
  fastq_2 = reverse_reads
) |> 
  mutate(
ArrayTaskID = row_number(),
    sample = basename(fastq_1), 
    sample = str_remove(sample, "_merged.*")
  ) |> 
  select(ArrayTaskID, sample, fastq_1, fastq_2) |> 
  as_tibble() 

# Print head
samples_table |> 
  head()

# A tibble: 2 × 4
  ArrayTaskID sample   fastq_1                                           fastq_2
        <int> <chr>    <chr>                                             <chr>  
1           1 MI-142-H /mnt/lustre/groups/maier/maide581/projects/Small… /mnt/l…
2           2 MI-237-H /mnt/lustre/groups/maier/maide581/projects/Small… /mnt/l…

In [8]:
# Write samples file
mifaser_samplesfile = file.path(sheets_dir, "Example_mifaser_samples.csv")
write_csv(samples_table, file = mifaser_samplesfile)

## Perform functional profiling

The functional profile can take hours, depending on the number of samples and
the sequencing depth, so we don't run it from the notebook. Instead we write a
small script stating what to run and which resources it needs, and hand it to
slurm, the cluster's scheduler, which starts it once a suitable machine is free.

In the following chunks, we'll generate the slurm scripts necessary to execute 
`mifaser`. We'll need to complete some fields and specify certain file names or
parameters.

In [9]:
mifaser_slurm_raw = str_glue(.open = "[", .close = "]",
"#!/bin/bash
##############################
#       Parameters           #
##############################

# This section tells the cluster what resources your job will need.
# These values are set in the notebook, in the chunk that fills this template.

# Name of the job
#SBATCH --job-name=[[job_name]]

# Generate an output file and give it a name
#SBATCH --output=%x-%j.out

# Number of tasks
#SBATCH --ntasks=1

# Number of cpus that this task will need
#SBATCH --cpus-per-task=[[cpu]]

# Specify the total memory required per node
#SBATCH --mem=[[memory]]

# Specify the maximum time this job can take to run before being killed (hh:mm:ss)
#SBATCH --time=23:59:59

# Specify number of array jobs
#SBATCH --array=[[array_jobs]]%[[simultaneous]]

# job information
scontrol show job ${SLURM_JOB_ID}

# per node
# prep
source $HOME/.bashrc

# Specify the path to the config file
samples_file=[[samples_file]]

# Extract the sample name for the current $SLURM_ARRAY_TASK_ID
sample=$(awk -F, -v ArrayTaskID=$SLURM_ARRAY_TASK_ID '$1==ArrayTaskID {print $2}' $samples_file)

# Extract the path to the forward read for the current $SLURM_ARRAY_TASK_ID
fastq_1=$(awk -F, -v ArrayTaskID=$SLURM_ARRAY_TASK_ID '$1==ArrayTaskID {print $3}' $samples_file)

# Extract the path to the reverse read for the current $SLURM_ARRAY_TASK_ID
fastq_2=$(awk -F, -v ArrayTaskID=$SLURM_ARRAY_TASK_ID '$1==ArrayTaskID {print $4}' $samples_file)

# Print to a file a message that includes the current $SLURM_ARRAY_TASK_ID and sample name
echo This is array task ${SLURM_ARRAY_TASK_ID}, the sample name is ${sample} the forward read is ${fastq_1} and the reverse is ${fastq_2}

# do your real computation
conda activate [[conda_env]]
cd [[mifaser_repo]]
python -m mifaser --lanes ${fastq_1} ${fastq_2} -o [[out_dir]]/${sample}_out -d GS-24-all -c [[cpu]]
")

Now we can replace the placeholders in the slurm script template with the actual
paths and filenames defined above. The parameters worth knowing are:

- `cpu` and `memory`: the processors and RAM given to each sample. The values below
  are a sensible start for gut metagenomes
- `array_jobs`: how many tasks the job is split into, one per sample. It is taken
  from the samples table, so you don't need to change it
- `simultaneous`: the most tasks that will run at the
  same time. It keeps a large submission from taking over the queue

In addition, in the slurm script it is specified that we will use the `GS-24-all`
database to assign EC numbers. This corresponds to the full database. See the 
`mifaser` documentation for a list of alternatives.

In [10]:
mifaser_slurm <- str_glue(
        mifaser_slurm_raw,
        job_name = "profiling_mifaser",
        array_jobs = str_c("1-", nrow(samples_table)),
        simultaneous = 50,
        cpu = "16",
        memory = "64G",
        samples_file = mifaser_samplesfile,
        mifaser_repo = mifaser_repo,
        out_dir = mifaser_dir,
        conda_env = conda_env,
        .open = "[",
        .close = "]"
)

mifaser_slurm |>
        print()

#!/bin/bash
##############################
#       Parameters           #
##############################

# This section tells the cluster what resources your job will need.
# These values are set in the notebook, in the chunk that fills this template.

# Name of the job
#SBATCH --job-name=profiling_mifaser

# Generate an output file and give it a name
#SBATCH --output=%x-%j.out

# Number of tasks
#SBATCH --ntasks=1

# Number of cpus that this task will need
#SBATCH --cpus-per-task=16

# Specify the total memory required per node
#SBATCH --mem=64G

# Specify the maximum time this job can take to run before being killed (hh:mm:ss)
#SBATCH --time=23:59:59

# Specify number of array jobs
#SBATCH --array=1-2%50

# job information
scontrol show job ${SLURM_JOB_ID}

# per node
# prep
source $HOME/.bashrc

# Specify the path to the config file
samples_file=/mnt/lustre/groups/maier/maide581/projects/Small_projects/Metemgee_remake/data/sheets/Example_mifaser_samples.csv

# Extract the sample na

In [11]:
# Write file
mifaser_slurmfile = file.path(sheets_dir, "mifaser_slurm.sh")
write_lines(mifaser_slurm, mifaser_slurmfile)

The following chunk prints the full command for you to copy and run in your terminal.

In [12]:
# Execution command
str_glue("cd {mifaser_dir} && sbatch {mifaser_slurmfile}")

cd /mnt/lustre/groups/maier/maide581/projects/Small_projects/Metemgee_remake/data/mifaser && sbatch /mnt/lustre/groups/maier/maide581/projects/Small_projects/Metemgee_remake/data/sheets/mifaser_slurm.sh

## While it runs

The job runs on its own, so you can close the notebook and your terminal.

- Expect a few hours, depending on how many samples you have and how deeply they
  were sequenced. Samples run in parallel, up to 50 at a time
- Check whether it's still running with `squeue`. If it's not listed, it's done
- Each task writes a log ending in `.out` in the folder you launched the job from.
  Look there if something failed

The chunk below stops the notebook on purpose. Everything from here on reads the
`mifaser` output, which doesn't exist until the cluster job has finished, so this
prevents "Run All" from running the merging steps too early.

Once the job is done, run the remaining chunks one by one, or comment out the `stop()`
and run the rest of the notebook.

In [13]:
stop("Downstream steps are to be done after mifaser finished executing")

: [1m[33mError[39m:[22m
[33m![39m Downstream steps are to be done after mifaser finished executing

## Merge tables
The output of `mifaser` is a table per sample. We will generate a single table
with all samples. The following assumes that each mifaser output folder is named
after the corresponding sample.

To generate a single merged table with annotations, run the following chunks:

In [14]:
# Obtain number of reads per sample
# Create a table
sequencing_depth = file.path(seq_dir, "Sequencing_depth.csv") |>
  read_csv(col_names = c("Sample", "File", "Depth")) |> 
  group_by(Sample) |> 
  slice(1) |> 
  ungroup() |> 
  select(Sample, Depth)

# Print head of table
sequencing_depth |> 
  head()


Rows: 4 Columns: 3
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (2): Sample, File
dbl (1): Depth

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


# A tibble: 2 × 2
  Sample      Depth
  <chr>       <dbl>
1 MI-142-H 30447298
2 MI-237-H 16443518

In [15]:
# Download EC annotation file
# Retrieved from HUMANn3 repo
ec_table = "https://github.com/biobakery/humann/raw/a9f181f32b3c66b66b73cabc611ff3ac55d87033/humann/data/utility_DEMO/map_level4ec_name.txt.gz" |> 
    read_tsv(col_names = c("EC_Number", "Annot"))

ec_table |> 
  head()

Rows: 7957 Columns: 2
── Column specification ────────────────────────────────────────────────────────
Delimiter: "\t"
chr (2): EC_Number, Annot

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


# A tibble: 6 × 2
  EC_Number Annot                                     
  <chr>     <chr>                                     
1 1.1.1.1   Alcohol dehydrogenase                     
2 1.1.1.10  L-xylulose reductase                      
3 1.1.1.100 3-oxoacyl-[acyl-carrier-protein] reductase
4 1.1.1.101 Acylglycerone-phosphate reductase         
5 1.1.1.102 3-dehydrosphinganine reductase            
6 1.1.1.103 L-threonine 3-dehydrogenase               

In [16]:
# Read output files and create a single table
ec_table_long = mifaser_dir |> 
    list.files(full.names = TRUE, recursive = TRUE, pattern = "analysis") |> 
    map(function(filename){
        # Name of sample
        sample_name = dirname(filename) |> 
            str_remove(fixed(mifaser_dir)) |> 
            str_remove("/") |> 
            str_remove("_out")
      sample_name
        
      # Read tables and add sample name
      # Skip first line (# unique mapped and multiple mapped reads)
        filename |> 
            read_tsv(skip = 1,col_names = c("EC_Number", "Count"), show_col_types = FALSE) |> 
            mutate(Sample = sample_name)
    }) |> 
  list_rbind() |> 
  left_join(ec_table, by = join_by(EC_Number)) |>
  select(Sample, EC_Number, Annot, Count) |> 
  left_join(sequencing_depth,  by = join_by(Sample)) |> 
  mutate(Relabund = Count/Depth) |> 
  relocate(Relabund, .after = Count)

# # Print head
ec_table_long |> 
  head()

# A tibble: 6 × 6
  Sample   EC_Number Annot                                 Count Relabund  Depth
  <chr>    <chr>     <chr>                                 <dbl>    <dbl>  <dbl>
1 MI-142-H 1.1.1.1   Alcohol dehydrogenase                  8404  2.76e-4 3.04e7
2 MI-142-H 1.1.1.2   Alcohol dehydrogenase (NADP(+))        1915  6.29e-5 3.04e7
3 MI-142-H 1.1.1.3   Homoserine dehydrogenase               1064  3.49e-5 3.04e7
4 MI-142-H 1.1.1.4   (R,R)-butanediol dehydrogenase          366  1.20e-5 3.04e7
5 MI-142-H 1.1.1.6   Glycerol dehydrogenase                 2652  8.71e-5 3.04e7
6 MI-142-H 1.1.1.8   Glycerol-3-phosphate dehydrogenase (…   457  1.50e-5 3.04e7

In [17]:
# Create wide table of counts
ec_table_counts = ec_table_long |> 
    pivot_wider(id_cols = c(EC_Number, Annot),
    names_from = Sample,   
    values_from = Count, 
    values_fill = 0)

# Print head
ec_table_counts |> 
  head()

# A tibble: 6 × 4
  EC_Number Annot                                       `MI-142-H` `MI-237-H`
  <chr>     <chr>                                            <dbl>      <dbl>
1 1.1.1.1   Alcohol dehydrogenase                             8404       7107
2 1.1.1.2   Alcohol dehydrogenase (NADP(+))                   1915        457
3 1.1.1.3   Homoserine dehydrogenase                          1064       1263
4 1.1.1.4   (R,R)-butanediol dehydrogenase                     366          5
5 1.1.1.6   Glycerol dehydrogenase                            2652       1243
6 1.1.1.8   Glycerol-3-phosphate dehydrogenase (NAD(+))        457        142

In [18]:
# Create wide table of relabund
ec_table_relabund = ec_table_long |> 
    pivot_wider(id_cols = c(EC_Number, Annot),
    names_from = Sample,   
    values_from = Relabund, 
    values_fill = 0)

# Print head
ec_table_relabund |> 
  head()

# A tibble: 6 × 4
  EC_Number Annot                                       `MI-142-H`  `MI-237-H`
  <chr>     <chr>                                            <dbl>       <dbl>
1 1.1.1.1   Alcohol dehydrogenase                        0.000276  0.000432   
2 1.1.1.2   Alcohol dehydrogenase (NADP(+))              0.0000629 0.0000278  
3 1.1.1.3   Homoserine dehydrogenase                     0.0000349 0.0000768  
4 1.1.1.4   (R,R)-butanediol dehydrogenase               0.0000120 0.000000304
5 1.1.1.6   Glycerol dehydrogenase                       0.0000871 0.0000756  
6 1.1.1.8   Glycerol-3-phosphate dehydrogenase (NAD(+))  0.0000150 0.00000864 

In [19]:
# Write table
# You can change the output directory or the name of the file if you wish
# By default it is located in the mifaser directory
counts_file = file.path(mifaser_dir, "Merged_mifaser_counts.tsv.gz")
write_tsv(ec_table_counts, counts_file)

relabund_file = file.path(mifaser_dir, "Merged_mifaser_relabund.tsv.gz")
write_tsv(ec_table_relabund, relabund_file)

long_file = file.path(mifaser_dir, "Merged_mifaser_long.tsv.gz")
write_tsv(ec_table_long, long_file)

## What you should have at the end

Inside `data/mifaser`, one folder per sample named `<sample>_out`, each containing:

- `analysis.tsv`, the EC numbers found in that sample and how many reads were assigned
  to each. Its first line is a comment with the number of uniquely and multiply mapped
  reads, which is why the chunks above skip it
- `ec_count.tsv`, which read was assigned to which EC number

The merging chunks then write three files to `data/mifaser`:

- `Merged_mifaser_counts.tsv.gz`: one row per EC number, one column per sample, raw counts
- `Merged_mifaser_relabund.tsv.gz`: the same table divided by the sequencing depth of each
  sample, which is what you want when comparing samples of different depth
- `Merged_mifaser_long.tsv.gz`: the same information in long format, one row per sample and
  EC number, with counts and relative abundances side by side